# ANRF AISEHack 2.0 — v14 (polyBERT + GNN + LGBM/XGB, 4-way blend)
 Same as v13 but the fine-tuned
transformer is **polyBERT** (`xushijie/polyBERT`) instead of ChemBERTa.

**Ensemble (per target):**
- Fingerprints + **fine-tuned polyBERT CLS embeddings** → **LGBM + XGB**
- Fine-tuned **polyBERT** regression head (predictions)
- Large **AttentiveFP GNN**
- Blended 4 ways (LGBM + XGB + GNN + polyBERT).

**Setup:** enable **GPU** and **Internet ON** (polyBERT downloads from Hugging Face).
Heavy run — budget a few hours; mind the 9-hour limit.


In [1]:
!pip install -q rdkit torch_geometric transformers

import os
import pandas as pd
import numpy as np
import glob, warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn.models import AttentiveFP

# Force anonymous HF access (Kaggle's implicit token can 401 on public repos).
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
for _k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN"):
    os.environ.pop(_k, None)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup, AutoConfig

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Base imports successful.')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_GPU = torch.cuda.is_available()
print(f'Torch device: {DEVICE}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.9 MB/s eta 0:00:00
Base imports successful.
Torch device: cuda


In [2]:
# ---- 1. DATA LOADING (Kaggle) ----
train_hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
test_hits  = glob.glob('/kaggle/input/**/test.csv',  recursive=True)
assert train_hits and test_hits, "train.csv/test.csv not found — add the competition dataset as input."
train = pd.read_csv(train_hits[0])
test  = pd.read_csv(test_hits[0])

train_tg  = train[train['target_type'] == 'tg'].reset_index(drop=True)
train_egc = train[train['target_type'] == 'egc'].reset_index(drop=True)
test_tg   = test[test['target_type'] == 'tg'].reset_index(drop=True)
test_egc  = test[test['target_type'] == 'egc'].reset_index(drop=True)

y_tg  = train_tg['target'].values
y_egc = train_egc['target'].values

print(f'Tg  train: {len(train_tg):,}  test: {len(test_tg):,}')
print(f'Egc train: {len(train_egc):,}  test: {len(test_egc):,}')

Tg  train: 4,143  test: 2,763
Egc train: 2,028  test: 1,352


In [3]:
# ---- 2. FEATURE ENGINEERING (RDKit, Morgan, MACCS) ----
DESC_NAMES  = [n for n, _ in Descriptors.descList]
MORGAN_ECFP4 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
MORGAN_ECFP6 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
RDKIT_FPGEN  = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)

def featurize(smiles_list):
    rdkit_rows, ecfp4_rows, ecfp6_rows, rdk_rows, maccs_rows = [], [], [], [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rdkit_rows.append([np.nan] * len(DESC_NAMES))
            ecfp4_rows.append(np.zeros(2048, dtype=np.uint8))
            ecfp6_rows.append(np.zeros(2048, dtype=np.uint8))
            rdk_rows.append(np.zeros(2048, dtype=np.uint8))
            maccs_rows.append(np.zeros(167, dtype=np.uint8))
        else:
            vals = Descriptors.CalcMolDescriptors(mol)
            rdkit_rows.append(list(vals.values()))
            ecfp4_rows.append(MORGAN_ECFP4.GetFingerprintAsNumPy(mol))
            ecfp6_rows.append(MORGAN_ECFP6.GetFingerprintAsNumPy(mol))
            rdk_rows.append(RDKIT_FPGEN.GetFingerprintAsNumPy(mol))
            maccs_rows.append(np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.uint8))
    return pd.concat([
        pd.DataFrame(rdkit_rows,  columns=DESC_NAMES),
        pd.DataFrame(ecfp4_rows,  columns=[f'ecfp4_{i}'  for i in range(2048)]),
        pd.DataFrame(ecfp6_rows,  columns=[f'ecfp6_{i}'  for i in range(2048)]),
        pd.DataFrame(rdk_rows,    columns=[f'rdkfp_{i}'  for i in range(2048)]),
        pd.DataFrame(maccs_rows,  columns=[f'maccs_{i}'  for i in range(167)]),
    ], axis=1)

def build_preprocessor(X_raw):
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.dropna(axis=1, thresh=int(0.2 * len(X)))
    X = X.loc[:, X.var() > 0]
    good_cols = X.columns.tolist()
    imputer  = SimpleImputer(strategy='median')
    X_imp    = imputer.fit_transform(X)
    scaler   = StandardScaler()
    return scaler.fit_transform(X_imp), (imputer, scaler, good_cols)

def apply_preprocessor(X_raw, preprocessor):
    imputer, scaler, good_cols = preprocessor
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.reindex(columns=good_cols, fill_value=np.nan)
    return scaler.transform(imputer.transform(X))

print('Featurizing tabular features...')
X_tg_raw       = featurize(train_tg['smiles'].tolist())
X_egc_raw      = featurize(train_egc['smiles'].tolist())
X_tg_test_raw  = featurize(test_tg['smiles'].tolist())
X_egc_test_raw = featurize(test_egc['smiles'].tolist())

X_tg,  tg_prep  = build_preprocessor(X_tg_raw)
X_egc, egc_prep = build_preprocessor(X_egc_raw)
X_tg_test  = apply_preprocessor(X_tg_test_raw,  tg_prep)
X_egc_test = apply_preprocessor(X_egc_test_raw, egc_prep)

Featurizing tabular features...


In [4]:
# ---- 3. POLYBERT FINE-TUNING (ensemble seeds + embedding extraction) ----
print('Preparing PolyBERT...')
model_name = "xushijie/polyBERT"

class SmilesDataset(Dataset):
    def __init__(self, smiles, tokenizer, targets=None, max_len=128):
        self.smiles = smiles; self.tokenizer = tokenizer; self.targets = targets; self.max_len = max_len
    def __len__(self):
        return len(self.smiles)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.smiles[idx], truncation=True, padding='max_length',
                             max_length=self.max_len, return_tensors='pt')
        item = {'input_ids': enc['input_ids'].squeeze(0), 'attention_mask': enc['attention_mask'].squeeze(0)}
        if self.targets is not None:
            item['labels'] = torch.tensor(self.targets[idx], dtype=torch.float)
        return item

TRANSFORMER_SEEDS = [42, 7]

def train_transformer_ensemble(df_train, df_test, y_true, target_name, seed, m_name, n_splits=5, epochs=15, batch_size=16, lr=2e-5):
    torch.manual_seed(seed); np.random.seed(seed)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_preds = np.zeros(len(df_train)); test_preds = np.zeros(len(df_test))
    config = AutoConfig.from_pretrained(m_name, token=False)
    hidden_dim = config.hidden_size
    tokenizer = AutoTokenizer.from_pretrained(m_name, token=False)
    oof_embs = np.zeros((len(df_train), hidden_dim)); test_embs = np.zeros((len(df_test), hidden_dim))
    test_loader = torch.utils.data.DataLoader(SmilesDataset(df_test['smiles'].tolist(), tokenizer), batch_size=batch_size, shuffle=False)

    for fold, (train_idx, val_idx) in enumerate(kf.split(df_train)):
        print(f"    PolyBERT fold {fold+1}/{n_splits} | {target_name} | Seed {seed}", flush=True)
        train_ds = SmilesDataset(df_train['smiles'].iloc[train_idx].tolist(), tokenizer, y_true[train_idx])
        val_ds   = SmilesDataset(df_train['smiles'].iloc[val_idx].tolist(),   tokenizer, y_true[val_idx])
        train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader   = torch.utils.data.DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
        val_y = y_true[val_idx]

        model = AutoModelForSequenceClassification.from_pretrained(m_name, num_labels=1, token=False).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        total_steps = len(train_loader) * epochs
        scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)
        criterion = nn.MSELoss()
        best_val_r2, best_state = -float('inf'), None

        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                ids = batch['input_ids'].to(DEVICE); am = batch['attention_mask'].to(DEVICE); lb = batch['labels'].to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(ids, attention_mask=am).logits.view(-1), lb)
                loss.backward(); optimizer.step(); scheduler.step()
            model.eval(); vp = []
            with torch.no_grad():
                for batch in val_loader:
                    ids = batch['input_ids'].to(DEVICE); am = batch['attention_mask'].to(DEVICE)
                    vp.extend(model(ids, attention_mask=am).logits.view(-1).cpu().numpy().tolist())
            r2v = r2_score(val_y, vp)
            if r2v > best_val_r2:
                best_val_r2 = r2v; best_state = {k: v.clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state); model.eval()
        vp, ve = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids = batch['input_ids'].to(DEVICE); am = batch['attention_mask'].to(DEVICE)
                out = model(ids, attention_mask=am, output_hidden_states=True)
                vp.extend(out.logits.view(-1).cpu().numpy().tolist())
                ve.extend(out.hidden_states[-1][:, 0, :].cpu().numpy())
        oof_preds[val_idx] = vp; oof_embs[val_idx] = np.array(ve)

        tp, te = [], []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE); am = batch['attention_mask'].to(DEVICE)
                out = model(ids, attention_mask=am, output_hidden_states=True)
                tp.extend(out.logits.view(-1).cpu().numpy().tolist())
                te.extend(out.hidden_states[-1][:, 0, :].cpu().numpy())
        test_preds += np.array(tp) / n_splits; test_embs += np.array(te) / n_splits

    print(f"  OOF PolyBERT R² ({target_name} Seed {seed}): {r2_score(y_true, oof_preds):.4f}")
    return oof_preds, test_preds, oof_embs, test_embs

print('='*60); print('  PolyBERT Fine-tuning & Embedding Extraction'); print('='*60)
p_oof_tg, p_test_tg, p_oofe_tg, p_teste_tg = [], [], [], []
for seed in TRANSFORMER_SEEDS:
    a,b,c,d = train_transformer_ensemble(train_tg, test_tg, y_tg, "Tg", seed, model_name)
    p_oof_tg.append(a); p_test_tg.append(b); p_oofe_tg.append(c); p_teste_tg.append(d)
oof_p_tg=np.mean(p_oof_tg,0); test_p_tg=np.mean(p_test_tg,0); oof_e_tg=np.mean(p_oofe_tg,0); test_e_tg=np.mean(p_teste_tg,0)

p_oof_e, p_test_e, p_oofe_e, p_teste_e = [], [], [], []
for seed in TRANSFORMER_SEEDS:
    a,b,c,d = train_transformer_ensemble(train_egc, test_egc, y_egc, "Egc", seed, model_name)
    p_oof_e.append(a); p_test_e.append(b); p_oofe_e.append(c); p_teste_e.append(d)
oof_p_egc=np.mean(p_oof_e,0); test_p_egc=np.mean(p_test_e,0); oof_e_egc=np.mean(p_oofe_e,0); test_e_egc=np.mean(p_teste_e,0)

Preparing PolyBERT...
  PolyBERT Fine-tuning & Embedding Extraction


config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

DebertaV2ForSequenceClassification LOAD REPORT from: xushijie/polyBERT
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight     | MISSING    | 
pooler.dense.bias       | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    PolyBERT fold 5/5 | Egc | Seed 7


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: xushijie/polyBERT
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight     | MISSING    | 
pooler.dense.bias       | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  OOF PolyBERT R² (Egc Seed 7): 0.8685


In [5]:
# ---- 4. GRAPH NEURAL NETWORK (AttentiveFP) ----
ATOM_LIST = ['C','N','O','S','F','Si','P','Cl','Br','I','B','*','Other']
def one_hot(v, ch):
    x=[0]*len(ch); x[ch.index(v) if v in ch else len(ch)-1]=1; return x
def atom_features(a):
    return one_hot(a.GetSymbol(), ATOM_LIST) + [a.GetDegree(), a.GetFormalCharge(),
            int(a.GetIsAromatic()), a.GetTotalNumHs(), int(a.GetHybridization())]
def bond_features(b):
    bt=b.GetBondType()
    return [int(bt==Chem.rdchem.BondType.SINGLE), int(bt==Chem.rdchem.BondType.DOUBLE),
            int(bt==Chem.rdchem.BondType.TRIPLE), int(bt==Chem.rdchem.BondType.AROMATIC),
            int(b.GetIsConjugated()), int(b.IsInRing())]
def smiles_to_graph(smiles, y=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for b in mol.GetBonds():
        i,j = b.GetBeginAtomIdx(), b.GetEndAtomIdx(); bf = bond_features(b)
        ei += [[i,j],[j,i]]; ea += [bf,bf]
    if not ei: ei, ea = [[0,0]], [[0]*6]
    d = Data(x=x, edge_index=torch.tensor(ei, dtype=torch.long).t().contiguous(),
             edge_attr=torch.tensor(ea, dtype=torch.float))
    if y is not None: d.y = torch.tensor([y], dtype=torch.float)
    return d

NODE_DIM = len(ATOM_LIST) + 5; EDGE_DIM = 6
y_tg_scaler = StandardScaler(); y_egc_scaler = StandardScaler()
y_tg_scaled  = y_tg_scaler.fit_transform(y_tg.reshape(-1,1)).ravel()
y_egc_scaled = y_egc_scaler.fit_transform(y_egc.reshape(-1,1)).ravel()

print('Building graphs...')
graphs_tg = [smiles_to_graph(s,y) for s,y in zip(train_tg['smiles'], y_tg_scaled)]
graphs_tg_test = [smiles_to_graph(s) for s in test_tg['smiles']]
graphs_egc = [smiles_to_graph(s,y) for s,y in zip(train_egc['smiles'], y_egc_scaled)]
graphs_egc_test = [smiles_to_graph(s) for s in test_egc['smiles']]

GNN_SEEDS = [42, 7]
def train_gnn_ensemble(graphs, y_raw, graphs_test, scaler, seed, n_splits=5,
                       hidden=128, layers=3, timesteps=3, epochs=150, patience=20, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_scaled = np.zeros(len(graphs)); test_scaled = np.zeros(len(graphs_test))
    test_loader = DataLoader(graphs_test, batch_size=256, shuffle=False)
    for fold, (tr, va) in enumerate(kf.split(np.arange(len(graphs))), 1):
        tl = DataLoader([graphs[i] for i in tr], batch_size=64, shuffle=True)
        vl = DataLoader([graphs[i] for i in va], batch_size=128, shuffle=False)
        model = AttentiveFP(in_channels=NODE_DIM, hidden_channels=hidden, out_channels=1,
                            edge_dim=EDGE_DIM, num_layers=layers, num_timesteps=timesteps, dropout=0.1).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5); loss_fn = nn.MSELoss()
        best, bad, state = float('inf'), 0, None
        for epoch in range(epochs):
            model.train()
            for batch in tl:
                batch = batch.to(DEVICE); opt.zero_grad()
                loss = loss_fn(model(batch.x, batch.edge_index, batch.edge_attr, batch.batch).squeeze(-1), batch.y)
                loss.backward(); opt.step()
            model.eval(); vloss=[]
            with torch.no_grad():
                for batch in vl:
                    batch = batch.to(DEVICE)
                    vloss.append(loss_fn(model(batch.x, batch.edge_index, batch.edge_attr, batch.batch).squeeze(-1), batch.y).item())
            v = np.mean(vloss)
            if v < best: best, bad, state = v, 0, {k:val.clone() for k,val in model.state_dict().items()}
            else:
                bad += 1
                if bad >= patience: break
        model.load_state_dict(state); model.eval()
        vp=[]
        with torch.no_grad():
            for batch in vl:
                batch = batch.to(DEVICE); vp.append(model(batch.x, batch.edge_index, batch.edge_attr, batch.batch).squeeze(-1).cpu().numpy())
        oof_scaled[va] = np.concatenate(vp)
        tp=[]
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(DEVICE); tp.append(model(batch.x, batch.edge_index, batch.edge_attr, batch.batch).squeeze(-1).cpu().numpy())
        test_scaled += np.concatenate(tp) / n_splits
    return (scaler.inverse_transform(oof_scaled.reshape(-1,1)).ravel(),
            scaler.inverse_transform(test_scaled.reshape(-1,1)).ravel())

print('='*60); print('  GNN (Large)'); print('='*60)
g_tg=[]; gt_tg=[]
for seed in GNN_SEEDS:
    print(f'  Tg seed={seed}')
    o,t = train_gnn_ensemble(graphs_tg, y_tg, graphs_tg_test, y_tg_scaler, seed=seed); g_tg.append(o); gt_tg.append(t)
oof_g_tg=np.mean(g_tg,0); test_g_tg=np.mean(gt_tg,0)
g_e=[]; gt_e=[]
for seed in GNN_SEEDS:
    print(f'  Egc seed={seed}')
    o,t = train_gnn_ensemble(graphs_egc, y_egc, graphs_egc_test, y_egc_scaler, seed=seed); g_e.append(o); gt_e.append(t)
oof_g_egc=np.mean(g_e,0); test_g_egc=np.mean(gt_e,0)

Building graphs...
  GNN (Large)
  Tg seed=42
  Tg seed=7
  Egc seed=42
  Egc seed=7


In [6]:
# ---- 5. TABULAR (LGBM + XGB) on fingerprints + polyBERT embeddings ----
SEEDS = [42, 7, 123]
def lgbm_params(t):
    p = dict(objective='regression', metric='rmse', n_estimators=4000, learning_rate=0.01,
             num_leaves=127, max_depth=-1, min_child_samples=15, subsample=0.8, subsample_freq=1,
             colsample_bytree=0.4, reg_alpha=0.05, reg_lambda=1.0, n_jobs=-1, verbose=-1)
    if t == 'egc': p['num_leaves'], p['min_child_samples'] = 63, 20
    return p
def xgb_params(t):
    p = dict(objective='reg:squarederror', n_estimators=4000, learning_rate=0.01, max_depth=6,
             min_child_weight=5, subsample=0.8, colsample_bytree=0.4, reg_alpha=0.05, reg_lambda=1.0,
             n_jobs=-1, tree_method='hist', early_stopping_rounds=200)
    if t == 'egc': p['max_depth'] = 5
    return p
def train_tabular_ensemble(X_train, y_train, X_test, t, seed, n_splits=5):
    X_train, y_train, X_test = np.asarray(X_train), np.asarray(y_train), np.asarray(X_test)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    ol, tl = np.zeros(len(X_train)), np.zeros(len(X_test))
    ox, tx = np.zeros(len(X_train)), np.zeros(len(X_test))
    lp = lgbm_params(t); lp['random_state'] = seed
    xp = xgb_params(t);  xp['random_state'] = seed
    for tr, va in kf.split(X_train):
        ml = lgb.LGBMRegressor(**lp)
        ml.fit(X_train[tr], y_train[tr], eval_set=[(X_train[va], y_train[va])],
               callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
        ol[va] = ml.predict(X_train[va]); tl += ml.predict(X_test) / n_splits
        mx = xgb.XGBRegressor(**xp); mx.fit(X_train[tr], y_train[tr], eval_set=[(X_train[va], y_train[va])], verbose=False)
        ox[va] = mx.predict(X_train[va]); tx += mx.predict(X_test) / n_splits
    return (ol, ox), (tl, tx)

print('='*60); print('  Tabular (LGBM+XGB) on Combined Features'); print('='*60)
X_tg_comb = np.hstack([X_tg, oof_e_tg]); X_tg_test_comb = np.hstack([X_tg_test, test_e_tg])
to, tt = [], []
for seed in SEEDS:
    print(f'  tabular Tg seed={seed} ...', flush=True)
    a,b = train_tabular_ensemble(X_tg_comb, y_tg, X_tg_test_comb, 'tg', seed=seed); to.append(a); tt.append(b)
oof_l_tg=np.mean([p[0] for p in to],0); oof_x_tg=np.mean([p[1] for p in to],0)
test_l_tg=np.mean([p[0] for p in tt],0); test_x_tg=np.mean([p[1] for p in tt],0)

X_egc_comb = np.hstack([X_egc, oof_e_egc]); X_egc_test_comb = np.hstack([X_egc_test, test_e_egc])
eo, et = [], []
for seed in SEEDS:
    print(f'  tabular Egc seed={seed} ...', flush=True)
    a,b = train_tabular_ensemble(X_egc_comb, y_egc, X_egc_test_comb, 'egc', seed=seed); eo.append(a); et.append(b)
oof_l_egc=np.mean([p[0] for p in eo],0); oof_x_egc=np.mean([p[1] for p in eo],0)
test_l_egc=np.mean([p[0] for p in et],0); test_x_egc=np.mean([p[1] for p in et],0)

  Tabular (LGBM+XGB) on Combined Features
  tabular Tg seed=42 ...
  tabular Tg seed=7 ...
  tabular Tg seed=123 ...
  tabular Egc seed=42 ...
  tabular Egc seed=7 ...
  tabular Egc seed=123 ...


In [7]:
# ---- 6/7. ENSEMBLE WEIGHTS + SUBMISSION ----
def find_best_weights(parts, y_true):
    stack = np.column_stack(parts)
    n = len(parts)
    res = minimize(lambda w: -r2_score(y_true, stack @ w), x0=[1/n]*n, method='SLSQP',
                   bounds=[(0,1)]*n, constraints={'type':'eq','fun':lambda w: w.sum()-1})
    return res.x

w_tg  = find_best_weights((oof_l_tg, oof_x_tg, oof_g_tg, oof_p_tg),   y_tg)
w_egc = find_best_weights((oof_l_egc, oof_x_egc, oof_g_egc, oof_p_egc), y_egc)

print('='*60); print('  v14 ENSEMBLE'); print('='*60)
print(f'Tg  weights — LGBM {w_tg[0]:.3f}  XGB {w_tg[1]:.3f}  GNN {w_tg[2]:.3f}  PolyBERT {w_tg[3]:.3f}')
print(f'Egc weights — LGBM {w_egc[0]:.3f}  XGB {w_egc[1]:.3f}  GNN {w_egc[2]:.3f}  PolyBERT {w_egc[3]:.3f}')

oof_tg_opt  = w_tg[0]*oof_l_tg + w_tg[1]*oof_x_tg + w_tg[2]*oof_g_tg + w_tg[3]*oof_p_tg
oof_egc_opt = w_egc[0]*oof_l_egc + w_egc[1]*oof_x_egc + w_egc[2]*oof_g_egc + w_egc[3]*oof_p_egc
r2_tg  = r2_score(y_tg, oof_tg_opt); r2_egc = r2_score(y_egc, oof_egc_opt)
print(f'\nOOF R² Tg: {r2_tg:.4f}  Egc: {r2_egc:.4f}  Mean: {(r2_tg+r2_egc)/2:.4f}')

pred_tg  = w_tg[0]*test_l_tg + w_tg[1]*test_x_tg + w_tg[2]*test_g_tg + w_tg[3]*test_p_tg
pred_egc = w_egc[0]*test_l_egc + w_egc[1]*test_x_egc + w_egc[2]*test_g_egc + w_egc[3]*test_p_egc

sub_tg = test_tg[['id']].copy(); sub_tg['target'] = pred_tg
sub_egc = test_egc[['id']].copy(); sub_egc['target'] = pred_egc
submission = pd.concat([sub_tg, sub_egc], axis=0).sort_values('id').reset_index(drop=True)
assert submission['target'].isna().sum() == 0
submission.to_csv('submission.csv', index=False)
print('submission.csv saved. rows:', len(submission))
print(submission.head().to_string())

  v14 ENSEMBLE
Tg  weights — LGBM 0.010  XGB 0.031  GNN 0.817  PolyBERT 0.142
Egc weights — LGBM 0.049  XGB 0.062  GNN 0.723  PolyBERT 0.166

OOF R² Tg: 0.9019  Egc: 0.9147  Mean: 0.9083
submission.csv saved. rows: 4115
   id      target
0   1  274.110601
1   2    4.767027
2   3   75.146352
3   4   46.318746
4   5   94.009460
